# ML Ensemble Lab — Bagging vs Boosting
**Dataset:** UCI Heart Disease — 4 centros combinados (`data2/heart_disease/processed.{cleveland,hungarian,switzerland,va}.data`, ~920 filas, 13 features clínicas). 

Objetivo: comparar **Bagging (RandomForest)** vs **Boosting (GradientBoosting / AdaBoost)** en **Clasificación** y **Regresión**.
- Clasificación: `Heart Disease Status` (Yes/No, derivado de la columna original `num`)
- Regresión: `Cholesterol Level` (continuo) + alternativa `Resting Blood Pressure`

In [13]:
import os, sys, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                               GradientBoostingClassifier, GradientBoostingRegressor, AdaBoostClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, RocCurveDisplay, mean_absolute_error, mean_squared_error,
                             r2_score, precision_recall_curve, average_precision_score, balanced_accuracy_score)
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

for candidate in ("../ml", "..", ".", "ml"):
    if os.path.exists(os.path.join(candidate, "pipeline.py")):
        sys.path.insert(0, os.path.abspath(candidate))
        break
else:
    raise FileNotFoundError("No se encontro pipeline.py; ajusta la ruta en 'candidate'.")

import pipeline as pl

TARGET_CLASS = pl.TARGET_CLASS
TARGET_REG = pl.TARGET_REG
TARGET_REG_ALT = pl.TARGET_REG_ALT

df = pl.load_data()
print("pipeline.py:", pl.__file__)
print("DATA_DIR:", pl.DATA_DIR)
print("Shape:", df.shape)

pipeline.py: c:\Users\julio\Downloads\INF_2_2026\ML\EXPOULTI\ml\pipeline.py
DATA_DIR: c:\Users\julio\Downloads\INF_2_2026\ML\EXPOULTI\data2\heart_disease
Shape: (920, 14)


## 1. Exploración del Dataset

In [14]:
df.head()

,Age,Sex,Chest Pain Type,Resting Blood Pressure,Cholesterol Level,Fasting Blood Sugar,Resting ECG,Max Heart Rate,Exercise Induced Angina,ST Depression,ST Slope,Major Vessels Colored,Thalassemia,Heart Disease Status
0,63.0,Male,Typical Angina,145.0,233.0,Yes,LV Hypertrophy,150.0,No,2.3,Downsloping,0.0,Fixed Defect,No
1,67.0,Male,Asymptomatic,160.0,286.0,No,LV Hypertrophy,108.0,Yes,1.5,Flat,3.0,Normal,Yes
2,67.0,Male,Asymptomatic,120.0,229.0,No,LV Hypertrophy,129.0,Yes,2.6,Flat,2.0,Reversible Defect,Yes
3,37.0,Male,Non-Anginal Pain,130.0,250.0,No,Normal,187.0,No,3.5,Downsloping,0.0,Normal,No
4,41.0,Female,Atypical Angina,130.0,204.0,No,LV Hypertrophy,172.0,No,1.4,Upsloping,0.0,Normal,No


In [15]:
df.dtypes

Age                        float64
Sex                         object
Chest Pain Type             object
Resting Blood Pressure     float64
Cholesterol Level          float64
Fasting Blood Sugar         object
Resting ECG                 object
Max Heart Rate             float64
Exercise Induced Angina     object
ST Depression              float64
ST Slope                    object
Major Vessels Colored      float64
Thalassemia                 object
Heart Disease Status        object
dtype: object

In [16]:
df.isnull().sum().sort_values(ascending=False).head(10)
# Nota: los NaN reflejan tanto "?" en los archivos originales como los
# ceros de Cholesterol Level reconvertidos a NaN en pipeline.load_data()

Major Vessels Colored      611
Thalassemia                486
ST Slope                   309
Cholesterol Level          202
Fasting Blood Sugar         90
ST Depression               62
Resting Blood Pressure      59
Max Heart Rate              55
Exercise Induced Angina     55
Resting ECG                  2
dtype: int64

In [17]:
df[TARGET_CLASS].value_counts(normalize=True)

Heart Disease Status
Yes    0.553261
No     0.446739
Name: proportion, dtype: float64

In [18]:
df[pl.NUMERIC_HINT].describe()

,Age,Resting Blood Pressure,Cholesterol Level,Max Heart Rate,ST Depression,Major Vessels Colored
count,920.000000,861.000000,718.000000,865.000000,858.000000,309.000000
mean,53.510870,132.132404,246.832869,137.545665,0.878788,0.676375
std,9.424685,19.066070,58.527062,25.926276,1.091226,0.935653
min,28.000000,0.000000,85.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,210.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,239.500000,140.000000,0.500000,0.000000
75%,60.000000,140.000000,276.750000,157.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,202.000000,6.200000,3.000000


In [19]:
plt.figure(figsize=(8,5))
sns.heatmap(df[pl.NUMERIC_HINT].corr(), cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Correlación — features numéricas")
plt.show()

In [20]:
for c in pl.CATEGORICAL_HINT:
    print(c, ":", df[c].nunique(), "->", df[c].unique()[:5])

Sex : 2 -> ['Male' 'Female']
Chest Pain Type : 4 -> ['Typical Angina' 'Asymptomatic' 'Non-Anginal Pain' 'Atypical Angina']
Fasting Blood Sugar : 2 -> ['Yes' 'No' nan]
Resting ECG : 3 -> ['LV Hypertrophy' 'Normal' 'ST-T Abnormality' nan]
Exercise Induced Angina : 2 -> ['No' 'Yes' nan]
ST Slope : 3 -> ['Downsloping' 'Flat' 'Upsloping' nan]
Thalassemia : 3 -> ['Fixed Defect' 'Normal' 'Reversible Defect' nan]


## 2. Preprocesamiento
- Numéricas: imputación mediana + StandardScaler
- Categóricas: imputación moda + OneHotEncoder

Se usa `pipeline.build_preprocessor`, la misma función que arma el `ColumnTransformer` en el backend Flask.

In [21]:
df_tmp = df.copy()
df_tmp[TARGET_CLASS] = df_tmp[TARGET_CLASS].map({"Yes": 1, "No": 0})
df_tmp = df_tmp.dropna(subset=[TARGET_CLASS])
X = df_tmp.drop(columns=[TARGET_CLASS]); y = df_tmp[TARGET_CLASS]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre, nc, cc = pl.build_preprocessor(pd.concat([X_train, X_test]), TARGET_CLASS)
print("Num:", nc)
print("Cat:", cc)

Num: ['Age', 'Resting Blood Pressure', 'Cholesterol Level', 'Max Heart Rate', 'ST Depression', 'Major Vessels Colored']
Cat: ['Sex', 'Chest Pain Type', 'Fasting Blood Sugar', 'Resting ECG', 'Exercise Induced Angina', 'ST Slope', 'Thalassemia']


## 3. Clasificación — Heart Disease Status
Comparativa Bagging (RandomForest) vs Boosting (GradientBoosting, AdaBoost)
### Métricas + Gráficos prioritarios

Nota: al combinar los 4 centros el desbalance de clases es más leve que si se usara un solo centro, pero igual conviene balancear para que la comparación Bagging vs Boosting sea justa. `RandomForestClassifier` balancea internamente vía `class_weight="balanced"`; `GradientBoostingClassifier`/`AdaBoostClassifier` no aceptan ese parámetro, así que se replica el mismo balanceo con `sample_weight` (igual que en `pipeline.py`).

In [22]:
def run_classification(n_estimators=100, lr=0.1):
    df2 = pl.load_data()
    Xtr, Xte, ytr, yte = pl.split_classification(df2, random_state=42)
    pre, nc, cc = pl.build_preprocessor(pd.concat([Xtr, Xte]), TARGET_CLASS)
    sample_weight = compute_sample_weight(class_weight="balanced", y=ytr)

    models = {
        "Bagging_RF": RandomForestClassifier(n_estimators=n_estimators, random_state=42, n_jobs=-1, class_weight="balanced"),
        "Boosting_GB": GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=lr, random_state=42),
        "Boosting_Ada": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=n_estimators, learning_rate=lr, random_state=42)
    }
    res = {}
    for name, clf in models.items():
        pipe = Pipeline([("pre", pre), ("clf", clf)])
        if name == "Bagging_RF":
            pipe.fit(Xtr, ytr)
        else:
            pipe.fit(Xtr, ytr, clf__sample_weight=sample_weight)
        yp = pipe.predict(Xte)
        yp_proba = pipe.predict_proba(Xte)[:, 1]
        res[name] = {"pipe": pipe, "yp": yp, "proba": yp_proba,
            "acc": accuracy_score(yte, yp), "prec": precision_score(yte, yp, zero_division=0),
            "rec": recall_score(yte, yp, zero_division=0), "f1": f1_score(yte, yp, zero_division=0),
            "roc": roc_auc_score(yte, yp_proba), "yte": yte}
    metrics = pd.DataFrame({k: {m: v for m, v in r.items() if m in ["acc", "prec", "rec", "f1", "roc"]} for k, r in res.items()}).T
    display(metrics.round(3))

    # 1) ROC Curves
    plt.figure(figsize=(5, 4))
    for n, r in res.items():
        RocCurveDisplay.from_predictions(r["yte"], r["proba"], name=n, ax=plt.gca())
    plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.title("ROC — Bagging vs Boosting"); plt.show()

    # 2) Matrices de confusión
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
    for i, (n, r) in enumerate(res.items()):
        sns.heatmap(confusion_matrix(r["yte"], r["yp"]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[i])
        ax[i].set_title(n); ax[i].set_xlabel("Pred"); ax[i].set_ylabel("Real")
    plt.tight_layout(); plt.show()

    # 3) Precision-Recall (mejor que ROC para clase minoritaria)
    plt.figure(figsize=(5, 4))
    for n, r in res.items():
        prec, rec, _ = precision_recall_curve(r["yte"], r["proba"])
        ap = average_precision_score(r["yte"], r["proba"])
        plt.plot(rec, prec, label=f"{n} (AP={ap:.3f})", lw=2)
    baseline = res[list(res.keys())[0]]["yte"].mean()
    plt.axhline(baseline, color="gray", ls="--", lw=1, label=f"Baseline ({baseline:.3f})")
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall"); plt.legend(); plt.show()

    # 4) Distribución de probabilidades (diagnostica AUC bajo)
    for n, r in res.items():
        plt.figure(figsize=(5, 4))
        sns.kdeplot(x=r["proba"][r["yte"] == 0], fill=True, alpha=0.35, label="Clase 0 (No)", color="#4575b4")
        sns.kdeplot(x=r["proba"][r["yte"] == 1], fill=True, alpha=0.35, label="Clase 1 (Yes)", color="#d73027")
        plt.axvline(0.5, color="black", ls="--", label="Umbral 0.5")
        plt.xlabel("Probabilidad predicha de Clase 1"); plt.title(f"Probabilidades - {n}"); plt.legend(); plt.show()

    # 5) Métricas vs Umbral (elige corte óptimo)
    thresholds = np.arange(0.05, 0.96, 0.05)
    plt.figure(figsize=(6, 4))
    for n, r in res.items():
        precs, recs, f1s = [], [], []
        for th in thresholds:
            y_pred_th = (r["proba"] >= th).astype(int)
            precs.append(precision_score(r["yte"], y_pred_th, zero_division=0))
            recs.append(recall_score(r["yte"], y_pred_th, zero_division=0))
            f1s.append(f1_score(r["yte"], y_pred_th, zero_division=0))
        plt.plot(thresholds, precs, "--", label=f"{n} Precision", alpha=0.7)
        plt.plot(thresholds, recs, "-.", label=f"{n} Recall", alpha=0.7)
        plt.plot(thresholds, f1s, "-", label=f"{n} F1", lw=2)
    plt.axvline(0.5, color="gray", ls=":", label="Umbral 0.5")
    plt.xlabel("Umbral"); plt.ylabel("Métrica"); plt.title("Métricas vs Umbral"); plt.legend(ncol=2); plt.grid(alpha=0.3); plt.show()

    # 6) Importancia por permutación (más robusta)
    for n in ["Bagging_RF", "Boosting_GB"]:
        pipe = res[n]["pipe"]
        result = permutation_importance(pipe, Xte, yte, scoring="average_precision", n_repeats=10, random_state=42, n_jobs=-1)
        cat_enc = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["onehot"]
        feat_names = nc + cat_enc.get_feature_names_out(cc).tolist()
        imp_mean = result.importances_mean
        idx = np.argsort(imp_mean)[::-1][:15]
        plt.figure(figsize=(6, 5))
        plt.barh(range(len(idx)), imp_mean[idx[::-1]], color="teal")
        plt.yticks(range(len(idx)), [feat_names[i] for i in idx[::-1]], fontsize=9)
        plt.xlabel("Disminución media en Average Precision")
        plt.title(f"Permutation Importance - {n}"); plt.tight_layout(); plt.show()

    # 7) Feature importance nativa (top 10)
    for n in ["Bagging_RF", "Boosting_GB"]:
        pipe = res[n]["pipe"]; pre_step = pipe.named_steps["pre"]
        cat_enc = pre_step.named_transformers_["cat"].named_steps["onehot"]
        feat_names = nc + cat_enc.get_feature_names_out(cc).tolist()
        imp = pipe.named_steps["clf"].feature_importances_
        idx = np.argsort(imp)[::-1][:10]
        plt.figure(figsize=(6, 3))
        sns.barplot(x=imp[idx], y=[feat_names[i] for i in idx], hue=[feat_names[i] for i in idx], legend=False)
        plt.title(f"Importancia nativa — {n}"); plt.tight_layout(); plt.show()

    return res, Xte, yte

res_cls, X_test_cls, y_test_cls = run_classification(n_estimators=100, lr=0.1)

,acc,prec,rec,f1,roc
Bagging_RF,0.799,0.804,0.843,0.823,0.910
Boosting_GB,0.810,0.807,0.863,0.834,0.900
Boosting_Ada,0.821,0.842,0.833,0.837,0.915


> Al combinar 4 centros (Cleveland, Hungría, Suiza, Long Beach VA) los patrones clínicos reales sí producen señal (AUC bien por encima de 0.5). Aun así **Suiza** y **Long Beach VA** aportan bastantes valores faltantes en columnas como `Major Vessels Colored` y `Thalassemia`, lo que limita el techo de las métricas. Se espera que **Bagging** dé resultados más estables entre corridas y que **Boosting** logre algo más de recall/F1 si se ajusta `learning_rate`, a costa de mayor sensibilidad al overfitting.

## 4. Regresión — Cholesterol Level (y alternativa Resting Blood Pressure)
Comparativa Bagging (RF Regressor) vs Boosting (GB Regressor)

In [23]:
def run_regression(target=TARGET_REG, n_estimators=100, lr=0.1):
    df2 = pl.load_data()
    Xtr, Xte, ytr, yte = pl.split_regression(df2, target_col=target, random_state=42)
    pre, nc, cc = pl.build_preprocessor(pd.concat([Xtr, Xte]), target)
    models = {
        "Bagging_RF": RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1),
        "Boosting_GB": GradientBoostingRegressor(n_estimators=n_estimators, learning_rate=lr, random_state=42)
    }
    res = {}
    for n, reg in models.items():
        pipe = Pipeline([("pre", pre), ("reg", reg)])
        pipe.fit(Xtr, ytr)
        yp = pipe.predict(Xte)
        res[n] = {"pipe": pipe, "yp": yp, "yte": yte,
                  "mae": mean_absolute_error(yte, yp), "mse": mean_squared_error(yte, yp),
                  "rmse": np.sqrt(mean_squared_error(yte, yp)), "r2": r2_score(yte, yp)}
    metrics = pd.DataFrame({k: {m: float(v) for m, v in r.items() if m in ["mae", "mse", "rmse", "r2"]} for k, r in res.items()}).T
    display(metrics.round(3))
    # Scatter Real vs Pred
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for i, (n, r) in enumerate(res.items()):
        ax[i].scatter(r["yte"], r["yp"], alpha=0.3, s=10)
        mn = min(r["yte"].min(), r["yp"].min()); mx = max(r["yte"].max(), r["yp"].max())
        ax[i].plot([mn, mx], [mn, mx], "r--"); ax[i].set_title(n); ax[i].set_xlabel("Real"); ax[i].set_ylabel("Pred")
    plt.suptitle(f"Real vs Predicho — {target}"); plt.tight_layout(); plt.show()
    # Residuales
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for i, (n, r) in enumerate(res.items()):
        ax[i].scatter(r["yp"], r["yte"] - r["yp"], alpha=0.3, s=10, color="darkred")
        ax[i].axhline(0, color="black", ls="--"); ax[i].set_title(n + " — Residuales"); ax[i].set_xlabel("Predicho"); ax[i].set_ylabel("Residual")
    plt.tight_layout(); plt.show()
    # Feature importance
    for n in res:
        pipe = res[n]["pipe"]; pre_step = pipe.named_steps["pre"]
        cat_enc = pre_step.named_transformers_["cat"].named_steps["onehot"]
        feat_names = nc + cat_enc.get_feature_names_out(cc).tolist()
        imp = pipe.named_steps["reg"].feature_importances_
        idx = np.argsort(imp)[::-1][:10]
        plt.figure(figsize=(6, 3))
        sns.barplot(x=imp[idx], y=[feat_names[i] for i in idx], hue=[feat_names[i] for i in idx], legend=False)
        plt.title(f"Importancia — {n} ({target})"); plt.tight_layout(); plt.show()
    return res

res_reg = run_regression(TARGET_REG, n_estimators=100, lr=0.1)
print("--- Alternativa: Resting Blood Pressure ---")
res_reg_alt = run_regression(TARGET_REG_ALT, n_estimators=100, lr=0.1)

,mae,mse,rmse,r2
Bagging_RF,17.801,514.732,22.688,0.252
Boosting_GB,17.569,500.595,22.374,0.272


--- Alternativa: Resting Blood Pressure ---


,mae,mse,rmse,r2
Bagging_RF,7.077,75.966,8.716,0.263
Boosting_GB,6.933,73.030,8.546,0.291


## 5. Comparativa Final — Tabla Resumen

In [24]:
print("Clasificación — F1 / ROC-AUC (Bagging vs Boosting)")
for k, v in res_cls.items():
    print(k, f"F1={v['f1']:.3f} ROC={v['roc']:.3f}")
print("\nRegresión Cholesterol Level — R2 / RMSE")
for k, v in res_reg.items():
    print(k, f"R2={v['r2']:.3f} RMSE={v['rmse']:.1f}")
print("\nRegresión Resting Blood Pressure — R2 / RMSE")
for k, v in res_reg_alt.items():
    print(k, f"R2={v['r2']:.3f} RMSE={v['rmse']:.2f}")

Clasificación — F1 / ROC-AUC (Bagging vs Boosting)
Bagging_RF F1=0.823 ROC=0.910
Boosting_GB F1=0.834 ROC=0.900
Boosting_Ada F1=0.837 ROC=0.915

Regresión Cholesterol Level — R2 / RMSE
Bagging_RF R2=0.252 RMSE=22.7
Boosting_GB R2=0.272 RMSE=22.4

Regresión Resting Blood Pressure — R2 / RMSE
Bagging_RF R2=0.263 RMSE=8.72
Boosting_GB R2=0.291 RMSE=8.55


## 6. Conclusiones para el Proyecto
- **Clasificación Heart Disease**: dataset real (4 centros UCI) con clases moderadamente desbalanceadas → se espera un AUC claramente por encima de 0.5. Con más datos y tuning, Boosting suele superar a Bagging en recall/F1 si se calibra `learning_rate` y el balanceo por `sample_weight`.
- **Regresión Cholesterol Level / Resting Blood Pressure**: son variables clínicas continuas con relaciones más débiles y ruidosas con el resto de features que un target diseñado para regresión; conviene comparar el R² contra un baseline (predecir la media) antes de sacar conclusiones fuertes. Sigue siendo útil para ilustrar **overfitting de Boosting** vs **estabilidad de Bagging**.

- **Gráficos generadospara mostrar en "demo"**
  1. Precision-Recall (mejor que ROC para clase minoritaria)
  2. Distribución de probabilidades (diagnostica AUC bajo)
  3. Métricas vs Umbral (elige corte óptimo ≠ 0.5)
  4. Importancia por permutación (scoring=average_precision)
  5. Matriz de confusión
  6. Boxplots por clase de `Age`, `Resting Blood Pressure`, `Cholesterol Level`, `Max Heart Rate`, `ST Depression`, `Major Vessels Colored`